# Дополнительное исследование

Три проверки, которые не вошли в основной ноутбук, чтобы не перегружать его.

1. Подбор параметров модели клика: как выбирались глубина дерева и скорость
   обучения.
2. Сравнение алгоритмов: действительно ли нужен градиентный бустинг или хватило
   бы более простой модели.
3. Вклад групп признаков: что дают контекст показа и история посетителя сверх
   характеристик самого предложения.

Данные берутся уже подготовленными, из основного ноутбука.

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.width", 180)

df = pd.read_parquet(Path.cwd().parent / "cache" / "prepared.parquet")

NUM_COLS = ["hour", "weekday", "weekend", "visit_depth",
            "visitor_shows", "visitor_clicks", "visitor_offer_shows", "since_last_show",
            "page_count", "ip_count", "ua_count",
            "ctr_offer", "ctr_link", "ctr_partner", "cr_link", "cr_partner",
            "payout_link", "payout_nominal"]
CAT_COLS = ["site_id", "placement_id", "position_id", "advertising_id", "advertising_type",
            "source_id", "campaign_id", "device_type", "os_name", "browser_name",
            "geo_country", "geo_region", "preset_id", "offer_id", "partner_offer_id", "partner_id"]
FEATURES = NUM_COLS + CAT_COLS

border = df["impression_at"].max().normalize() - pd.Timedelta(days=13)
train, test = df[df["impression_at"] < border], df[df["impression_at"] >= border]
y_test = test["target_click"].to_numpy()
print(f"обучение {len(train):,}, проверка {len(test):,}".replace(",", " "))

обучение 1 736 724  проверка 918 447


## 1. Подбор параметров

Проверять параметры на отложенной выборке нельзя: она нужна для итоговой оценки, и
если подбирать по ней, результат окажется завышенным.

Поэтому внутри обучающего периода делаю три проверочных окна по неделе: модель
учится на том, что было раньше окна, и проверяется на самом окне. Такое разбиение
повторяет реальную ситуацию, когда модель учится на прошлом и работает на будущем.

In [2]:
windows = []
end = train["impression_at"].max().normalize()
for i in range(3, 0, -1):
    hi = end - pd.Timedelta(days=7 * (i - 1))
    lo = hi - pd.Timedelta(days=7)
    windows.append((train["impression_at"] < lo,
                    (train["impression_at"] >= lo) & (train["impression_at"] < hi)))

variants = [{"depth": 6, "learning_rate": 0.1}, {"depth": 8, "learning_rate": 0.08}]
results = []

for params in variants:
    scores, trees = [], []
    for fit_mask, val_mask in windows:
        model = CatBoostClassifier(iterations=500, eval_metric="AUC", random_seed=42,
                                   early_stopping_rounds=60, verbose=False, **params)
        model.fit(Pool(train[fit_mask][FEATURES], train[fit_mask]["target_click"], cat_features=CAT_COLS),
                  eval_set=Pool(train[val_mask][FEATURES], train[val_mask]["target_click"],
                                cat_features=CAT_COLS))
        scores.append(model.get_best_score()["validation"]["AUC"])
        trees.append(model.get_best_iteration())
    results.append({**params, "ROC-AUC": np.mean(scores), "деревьев": int(np.mean(trees))})
    print(f"глубина {params['depth']}, шаг {params['learning_rate']}: "
          f"AUC {np.mean(scores):.4f}, деревьев {int(np.mean(trees))}")

pd.DataFrame(results).round(4)

глубина 6, шаг 0.1: AUC 0.7242, деревьев 208


глубина 8, шаг 0.08: AUC 0.7301, деревьев 281


,depth,learning_rate,ROC-AUC,деревьев
0,6,0.10,0.7242,208
1,8,0.08,0.7301,281


Вариант с большей глубиной выигрывает, но разница в третьем знаке. В основном
ноутбуке беру более простой: при почти одинаковом качестве меньшая глубина
обучается быстрее и меньше рискует переобучиться. Число деревьев беру с
небольшим запасом, потому что финальная модель учится на большем объёме данных,
чем каждое отдельное окно.

## 2. Сравнение алгоритмов

Логистическая регрессия и случайный лес не работают с категориями напрямую: им
нужно превратить каждую категорию в набор из нулей и единиц. Для дробных
категорий, вроде размещения или региона, это даёт слишком много колонок, поэтому
такие признаки им приходится не давать вовсе.

Это само по себе часть ответа на вопрос, зачем здесь бустинг.

In [3]:
SIMPLE_CATS = [c for c in CAT_COLS if df[c].nunique() <= 70]
print("категорий отдаём простым моделям:", len(SIMPLE_CATS), "из", len(CAT_COLS))
print("не влезли:", [c for c in CAT_COLS if c not in SIMPLE_CATS])

prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=50), SIMPLE_CATS),
])

compare = []

logreg = Pipeline([("prep", prep), ("model", LogisticRegression(max_iter=300, n_jobs=-1))])
logreg.fit(train[NUM_COLS + SIMPLE_CATS], train["target_click"])
p = logreg.predict_proba(test[NUM_COLS + SIMPLE_CATS])[:, 1]
compare.append(("логистическая регрессия", roc_auc_score(y_test, p), average_precision_score(y_test, p)))
print("регрессия обучена")

forest = Pipeline([("prep", prep),
                   ("model", RandomForestClassifier(n_estimators=120, max_depth=16,
                                                    min_samples_leaf=50, n_jobs=-1, random_state=42))])
forest.fit(train[NUM_COLS + SIMPLE_CATS], train["target_click"])
p = forest.predict_proba(test[NUM_COLS + SIMPLE_CATS])[:, 1]
compare.append(("случайный лес", roc_auc_score(y_test, p), average_precision_score(y_test, p)))
print("лес обучен")

boosting = CatBoostClassifier(depth=6, learning_rate=0.1, iterations=318, random_seed=42, verbose=False)
boosting.fit(Pool(train[FEATURES], train["target_click"], cat_features=CAT_COLS))
p = boosting.predict_proba(test[FEATURES])[:, 1]
compare.append(("градиентный бустинг", roc_auc_score(y_test, p), average_precision_score(y_test, p)))

pd.DataFrame(compare, columns=["модель", "ROC-AUC", "PR-AUC"]).set_index("модель").round(4)

категорий отдаём простым моделям: 12 из 16
не влезли: ['placement_id', 'campaign_id', 'geo_country', 'geo_region']


регрессия обучена


лес обучен


,ROC-AUC,PR-AUC
модель,,
логистическая регрессия,0.7187,0.1497
случайный лес,0.7508,0.1840
градиентный бустинг,0.7538,0.1923


## 3. Вклад групп признаков

Три набора, вложенных друг в друга: сначала только характеристики предложения,
потом добавляется контекст показа, потом история посетителя. Модель и параметры
одинаковые, меняется только состав признаков.

In [4]:
OFFER_NUM = ["ctr_offer", "ctr_link", "ctr_partner", "cr_link", "cr_partner",
             "payout_link", "payout_nominal"]
OFFER_CAT = ["preset_id", "offer_id", "partner_offer_id", "partner_id"]
CONTEXT_NUM = ["hour", "weekday", "weekend", "visit_depth"]
CONTEXT_CAT = [c for c in CAT_COLS if c not in OFFER_CAT]
VISITOR_NUM = ["visitor_shows", "visitor_clicks", "visitor_offer_shows", "since_last_show",
               "page_count", "ip_count", "ua_count"]

feature_sets = {
    "только предложение": (OFFER_NUM, OFFER_CAT),
    "плюс контекст показа": (OFFER_NUM + CONTEXT_NUM, OFFER_CAT + CONTEXT_CAT),
    "плюс история посетителя": (OFFER_NUM + CONTEXT_NUM + VISITOR_NUM, OFFER_CAT + CONTEXT_CAT),
}

rows = []
for name, (nums, cats) in feature_sets.items():
    cols = nums + cats
    model = CatBoostClassifier(depth=6, learning_rate=0.1, iterations=318, random_seed=42, verbose=False)
    model.fit(Pool(train[cols], train["target_click"], cat_features=cats))
    p = model.predict_proba(test[cols])[:, 1]
    rows.append((name, len(cols), roc_auc_score(y_test, p), average_precision_score(y_test, p)))
    print(f"{name}: {len(cols)} признаков, AUC {roc_auc_score(y_test, p):.4f}")

pd.DataFrame(rows, columns=["набор признаков", "сколько", "ROC-AUC", "PR-AUC"]).set_index("набор признаков").round(4)

только предложение: 11 признаков, AUC 0.6743


плюс контекст показа: 27 признаков, AUC 0.7493


плюс история посетителя: 34 признаков, AUC 0.7527


,сколько,ROC-AUC,PR-AUC
набор признаков,,,
только предложение,11,0.6743,0.1374
плюс контекст показа,27,0.7493,0.1825
плюс история посетителя,34,0.7527,0.1912


## Выводы

**Параметры.** Разница между вариантами невелика, поэтому взял более простой:
меньшая глубина при том же качестве означает меньше риска переобучения.

**Алгоритмы.** Бустинг лучше остальных, но разрыв со случайным лесом небольшой.
Главное преимущество не в самих числах, а в том, что бустинг работает с
категориями напрямую: регрессии и лесу пришлось отдать неполный набор данных.

**Признаки.** Основной прирост даёт контекст показа. Значит задача действительно
про то, кому и где показать, а не только про то, что показать. История посетителя
добавляет меньше, и это объяснимо: внутри одной витрины она одинакова у всех
карточек, поэтому помогает оценить показ целиком, но не помогает выбрать лучшую
карточку из нескольких.